##
 Nesse notebook "Exploração de alarme", faz o DESCRIBE da Silver, perfila alarme (valores, nulos, taxa global, por dia, hora e dispositivo) e cruza com as 3 flags usando o lift. Termina com uma célula de conclusão para você preencher e definir o peso.

In [0]:
CATALOG = "project_lakehouse_telemetria_ele"
SILVER = f"{CATALOG}.silver.telemetria_clean"
 
# Nomes de colunas. Confira com o DESCRIBE abaixo e ajuste se necessário.
C_DEV, C_TS, C_ALARM = "device_id", "event_ts", "alarme"
F_PICO, F_TENS, F_SENT = "temp_pico", "tensao_ausente", "corrente_sentinela"
 
display(spark.sql(f"DESCRIBE {SILVER}"))

In [0]:
# 1. Perfil da coluna
 
display(spark.sql(f"""
SELECT COUNT(*) AS linhas,
       SUM(CASE WHEN {C_ALARM} IS NULL THEN 1 ELSE 0 END) AS nulos,
       ROUND(100*AVG(CASE WHEN {C_ALARM} IS NULL THEN 1 ELSE 0 END),3) AS pct_nulos,
       COUNT(DISTINCT {C_ALARM}) AS valores_distintos
FROM {SILVER}
"""))

In [0]:
display(spark.sql(f"""
SELECT {C_ALARM} AS valor, COUNT(*) AS linhas,
       ROUND(100*COUNT(*)/SUM(COUNT(*)) OVER (),3) AS pct
FROM {SILVER}
GROUP BY {C_ALARM} ORDER BY linhas DESC LIMIT 20
"""))

In [0]:
ALARME_ATIVO = f"""(CASE WHEN {C_ALARM} = 'SOBRETEMP' THEN 1 ELSE 0 END)"""   # valores reais: OK | SOBRETEMP
 
def fi(c):  # flag como 0/1, nulo vira 0
    return f"COALESCE(CAST({c} AS INT),0)"
 
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v_base AS
SELECT *, {ALARME_ATIVO} AS alarme_ativo,
       {fi(F_PICO)} AS f_pico, {fi(F_TENS)} AS f_tens, {fi(F_SENT)} AS f_sent
FROM {SILVER}
""")

In [0]:
## 2. Taxa de alarme: global, por dia, por hora
  
display(spark.sql("""
SELECT COUNT(*) AS linhas, SUM(alarme_ativo) AS alarmes,
       ROUND(100*AVG(alarme_ativo),3) AS pct_alarme
FROM v_base
"""))

In [0]:
display(spark.sql("""
SELECT data_evento, COUNT(*) AS linhas, SUM(alarme_ativo) AS alarmes,
       ROUND(100*AVG(alarme_ativo),3) AS pct_alarme
FROM v_base GROUP BY data_evento ORDER BY data_evento
"""))

In [0]:
display(spark.sql(f"""
SELECT HOUR({C_TS}) AS hora, COUNT(*) AS linhas, SUM(alarme_ativo) AS alarmes,
       ROUND(100*AVG(alarme_ativo),3) AS pct_alarme
FROM v_base GROUP BY HOUR({C_TS}) ORDER BY hora
"""))

In [0]:
## 3. Distribuição por dispositivo
# Se a taxa varia pouco entre os 50 dispositivos, o alarme discrimina mal.
 
display(spark.sql(f"""
SELECT {C_DEV} AS device_id, COUNT(*) AS linhas, SUM(alarme_ativo) AS alarmes,
       ROUND(100*AVG(alarme_ativo),3) AS pct_alarme
FROM v_base GROUP BY {C_DEV} ORDER BY alarmes DESC, device_id LIMIT 15
"""))

In [0]:
display(spark.sql(f"""
WITH d AS (SELECT {C_DEV} AS device_id, AVG(alarme_ativo) AS taxa FROM v_base GROUP BY {C_DEV})
SELECT COUNT(*) AS dispositivos,
       ROUND(100*MIN(taxa),3) AS min_pct, ROUND(100*AVG(taxa),3) AS media_pct,
       ROUND(100*MAX(taxa),3) AS max_pct, ROUND(100*STDDEV(taxa),4) AS desvio_pct,
       ROUND(STDDEV(taxa)/NULLIF(AVG(taxa),0),3) AS coef_variacao
FROM d
"""))

In [0]:
## 4. Cruzamentos com as flags da Silver
#`lift` = taxa de alarme com a flag ÷ taxa sem a flag. Perto de 1 = independente; bem acima de 1 = alarme acompanha o problema.
 
# COMMAND ----------
 
display(spark.sql("""
SELECT flag, COUNT(CASE WHEN v=1 THEN 1 END) AS linhas_com_flag,
       ROUND(100*AVG(CASE WHEN v=1 THEN alarme_ativo END),3) AS pct_alarme_com_flag,
       ROUND(100*AVG(CASE WHEN v=0 THEN alarme_ativo END),3) AS pct_alarme_sem_flag,
       ROUND(AVG(CASE WHEN v=1 THEN alarme_ativo END)
             / NULLIF(AVG(CASE WHEN v=0 THEN alarme_ativo END),0),2) AS lift
FROM (
  SELECT 'temp_pico' AS flag, f_pico AS v, alarme_ativo FROM v_base
  UNION ALL SELECT 'tensao_ausente', f_tens, alarme_ativo FROM v_base
  UNION ALL SELECT 'corrente_sentinela', f_sent, alarme_ativo FROM v_base
) GROUP BY flag
"""))

In [0]:
## 5. Alarme × pico: sobreposição exata

display(spark.sql("""
SELECT alarme_ativo, f_pico, COUNT(*) AS linhas FROM v_base
GROUP BY alarme_ativo, f_pico ORDER BY alarme_ativo, f_pico
"""))

In [0]:
# 6. Os outros sinais variam entre dispositivos?
# Coeficiente de variação da taxa por dispositivo. Perto de 0 = ruído uniforme, sem poder de separação.

display(spark.sql("""
WITH d AS (SELECT device_id, AVG(f_pico) AS picos, AVG(f_tens) AS tensao, AVG(f_sent) AS sentinela
           FROM v_base GROUP BY device_id)
SELECT 'picos' AS sinal, ROUND(100*AVG(picos),3) AS media_pct, ROUND(STDDEV(picos)/AVG(picos),3) AS coef_variacao FROM d
UNION ALL SELECT 'tensao_ausente', ROUND(100*AVG(tensao),3), ROUND(STDDEV(tensao)/AVG(tensao),3) FROM d
UNION ALL SELECT 'corrente_sentinela', ROUND(100*AVG(sentinela),3), ROUND(STDDEV(sentinela)/AVG(sentinela),3) FROM d
"""))

## 5. Conclusão (preencher)
#  Taxa global de alarme: ___
#  Varia entre dispositivos? (coef. de variação): ___
#  Lift com `temp_pico`: 3150 → alarme ( forte / fraco / redundante )
#  Peso sugerido no score: ___ (ponto de partida 0,4; reduzir se for redundante com picos ou quase constante entre dispositivos)